# EDA

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('customers_clean.csv')
df.head()

In [ ]:
print("Shape is: ", df.shape)

In [ ]:
df.info()

In [ ]:
sns.set_style('whitegrid')

plt.figure(figsize=(6,4))
ax = sns.countplot(data=df, x='Churn',hue='Churn', palette=['#2ecc71', '#e74c3c'], legend=False)

plt.title('Customer Churn Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Churn')
plt.ylabel('Number of Customers')

for container in ax.containers:
    ax.bar_label(container)

plt.savefig('images/customer-churn.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
contract_churn = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100

contract_churn.plot(kind='bar', stacked=False, figsize=(7,5), color=['#4C72B0', '#DD8452'])

plt.title('Churn Rate by Contract Type (%)')
plt.xlabel('Contract Type')
plt.ylabel('Percentage of Customers')
plt.xticks(rotation=0)
plt.legend(title='Churn')
plt.tight_layout()
plt.savefig('images/churn-rate-by-contract-type.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.kdeplot(data=df, x='tenure', hue='Churn', fill=True, 
            common_norm=False, alpha=0.4, linewidth=2,
            clip=(0, df['tenure'].max()))

plt.title('Tenure Distribution: Churned vs Retained Customers')
plt.xlabel('Tenure (months)')
plt.ylabel('Density')
plt.tight_layout()
plt.show()

In [ ]:
internet_churn = pd.crosstab(df['InternetService'], df['Churn'], normalize='index') * 100

internet_churn.plot(kind='bar', stacked=False, figsize=(7,5), color=['#4C72B0', '#DD8452'])

plt.title('Churn Rate by Internet Service Type (%)')
plt.xlabel('Internet Service')
plt.ylabel('Percentage of Customers')
plt.xticks(rotation=0)
plt.legend(title='Churn')
plt.tight_layout()
plt.savefig('images/churn-rate-by-internet-service-type.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df_corr = df.copy()
df_corr['Churn_numeric'] = df_corr['Churn'].map({'Yes': 1, 'No': 0})

numeric_cols = df_corr.select_dtypes(include=['int64', 'float64']).columns
corr_matrix = df_corr[numeric_cols].corr()

# Plot heatmap
plt.figure(figsize=(7,6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Correlation Heatmap: Numeric Features vs Churn')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7,5))

sns.boxplot(data=df, x='Churn', y='MonthlyCharges', palette=['#4C72B0', '#DD8452'])

plt.title('Monthly Charges Distribution by Churn Status')
plt.xlabel('Churn')
plt.ylabel('Monthly Charges ($)')
plt.tight_layout()
plt.savefig('images/monthly-charges-by-churn-status.png', dpi=150, bbox_inches='tight')
plt.show()

# Feature Engineering

In [ ]:
# a clean backup before transforming the data
df_backup = df.copy()

In [ ]:
pd.set_option("display.max_colwidth", None)

unique_values = df.select_dtypes(include='object').drop(columns='customerID').apply(lambda col: col.unique())
print(unique_values)

In [ ]:
# Columns that contain the redundant "No internet service" value
internet_dependent_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                            'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in internet_dependent_cols:
    df[col] = df[col].replace('No internet service', 'No')

df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

# Confirm each column now has only 2 unique values
for col in internet_dependent_cols + ['MultipleLines']:
    print(f"{col}: {df[col].unique()}")

In [ ]:
df = df.drop(columns='customerID')

print(df.columns.tolist())

In [ ]:
df['estimated_total'] = df['tenure'] * df['MonthlyCharges']
comparison = df[['TotalCharges', 'estimated_total']].head(10)
print(comparison)

df['diff_pct'] = ((df['TotalCharges'] - df['estimated_total']).abs() / df['TotalCharges']) * 100
print(f"\nAverage difference: {df['diff_pct'].mean():.2f}%")

# Clean up temp columns
df = df.drop(columns=['estimated_total', 'diff_pct'])

In [ ]:
# Drop TotalCharges — confirmed as ~97% derivable from tenure * MonthlyCharges (3.21% avg diff)
# and highly collinear with tenure (0.83 correlation)
df = df.drop(columns='TotalCharges')

print(df.columns.tolist())
print("Remaining columns: ", {df.shape[1]})

### Binary Encoder

In [ ]:
# Columns with Yes/No values
yes_no_cols = ['Churn', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling',
               'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
               'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in yes_no_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

print(df.dtypes)
print(df[['Churn', 'gender', 'Partner']].head())

In [ ]:
# One-Hot Encode the remaining multi-category columns
df = pd.get_dummies(df, columns=['InternetService', 'Contract', 'PaymentMethod'], 
                     drop_first=True)

print(df.dtypes)
print(df.shape)

# Building Machine Learning Models

### Preparing the Data for the Models

In [ ]:
X = df.drop(columns='Churn')
y = df['Churn']

print(X.columns.tolist())

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nX columns:")
print(X.columns.tolist())
print("\ny value counts:")
print(y.value_counts())

In [ ]:
# Train Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

In [ ]:
# Standard Scaler
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[['tenure', 'MonthlyCharges']] = scaler.fit_transform(X_train[['tenure', 'MonthlyCharges']])
X_test[['tenure', 'MonthlyCharges']] = scaler.transform(X_test[['tenure', 'MonthlyCharges']])

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

lr = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr.fit(X_train, y_train)

In [ ]:
y_pred_lr = lr.predict(X_test)

In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred_lr))

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100)
rf.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf.predict(X_test)

In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred_rf))

### XGBoost

In [ ]:
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, 
                            eval_metric='logloss')
xgb_model.fit(X_train, y_train)

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)

In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred_xgb))

### SVM

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(class_weight='balanced', random_state=42, probability=True)
svm_model.fit(X_train, y_train)

In [ ]:
y_pred_svm = svm_model.predict(X_test)

In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred_svm))

# Hyperparameter Tuning

### Random Forest

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define the grid of hyperparameters to try
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', 'balanced_subsample']
}

rf_base = RandomForestClassifier(random_state=42)

grid_search_rf = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_rf.fit(X_train, y_train)

print("Best parameters:", grid_search_rf.best_params_)
print("Best F1-score (cross-validated):", grid_search_rf.best_score_)

In [ ]:
best_rf = grid_search_rf.best_estimator_

y_pred_rf_tuned = best_rf.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_tuned))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf_tuned))

### XGBoost

In [ ]:
from xgboost import XGBClassifier

# Define the grid of hyperparameters for XGBoost
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'scale_pos_weight': [1, scale_pos_weight]  # try both unweighted and imbalance-corrected
}

xgb_base = XGBClassifier(random_state=42, eval_metric='logloss')

grid_search_xgb = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_xgb,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_xgb.fit(X_train, y_train)

print("Best parameters:", grid_search_xgb.best_params_)
print("Best F1-score (cross-validated):", grid_search_xgb.best_score_)

In [ ]:
# Get the best XGBoost model found by GridSearch
best_xgb = grid_search_xgb.best_estimator_

y_pred_xgb_tuned = best_xgb.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb_tuned))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb_tuned))

# Feature Importance from XGBoost

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Extract feature importances from the tuned XGBoost model
importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': best_xgb.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(importances)

# Plot top 15 features
plt.figure(figsize=(8,6))
plt.barh(importances['Feature'][:15][::-1], importances['Importance'][:15][::-1])
plt.title('Top 15 Feature Importances (Tuned XGBoost)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('images/Top-15-Feature-Importances-Tuned-XGBoost.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import joblib
joblib.dump(best_xgb, 'xgb_churn_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(X_train.columns.tolist(), 'feature_columns.pkl')

In [ ]:
import joblib
import numpy as np

# Load saved artifacts
loaded_model = joblib.load("xgb_churn_model.pkl")
loaded_scaler = joblib.load("scaler.pkl")
loaded_features = joblib.load("feature_columns.pkl")

# 1. Check feature columns
print("Number of saved features:", len(loaded_features))
print("Number of training features:", X_train.shape[1])
print("Features match:", loaded_features == X_train.columns.tolist())

# 2. Check scaler configuration
print("\nScaler feature count:", loaded_scaler.n_features_in_)
print("Scaler feature names:", getattr(loaded_scaler, "feature_names_in_", "Not stored"))

# 3. Check model feature count
print("\nModel feature count:", loaded_model.n_features_in_)

# 4. Check prediction using saved model
sample = X_test.iloc[:5].copy()

sample_scaled = sample.copy()
sample_scaled[["tenure", "MonthlyCharges"]] = loaded_scaler.transform(
    sample[["tenure", "MonthlyCharges"]]
)

predictions = loaded_model.predict(sample_scaled)

print("\nSample predictions:", predictions)
print("Compatibility check completed.")

In [ ]:
import numpy as np

# Use the same test samples
sample = X_test.iloc[:20].copy()

# Apply the saved scaler to the two numerical columns
sample_scaled = sample.copy()

sample_scaled[["tenure", "MonthlyCharges"]] = loaded_scaler.transform(
    sample[["tenure", "MonthlyCharges"]]
)

# Predictions from the loaded model
saved_predictions = loaded_model.predict(sample_scaled)

# Predictions from the original model
original_predictions = best_xgb.predict(sample_scaled)

# Compare predictions
print(
    "Predictions match:",
    np.array_equal(saved_predictions, original_predictions)
)

print("Saved model predictions:", saved_predictions)
print("Original model predictions:", original_predictions)